In [2]:
# yolo格式标注构建
import os
import xml.etree.ElementTree as ET

# 读取类别
classes_path = 'helmet/classes.txt'
with open(classes_path, 'r') as f:
    classes = [line.strip() for line in f.readlines()]
class_to_id = {cls: idx for idx, cls in enumerate(classes)}
print(f'类别映射: {class_to_id}')

# 创建labels目录
labels_dir = 'helmet/labels'
os.makedirs(labels_dir, exist_ok=True)

# 转换标注 Pascal VOC XML -> YOLO TXT
annotations_dir = 'helmet/annotations'
xml_files = [f for f in os.listdir(annotations_dir) if f.endswith('.xml')]

for xml_file in xml_files:
    tree = ET.parse(os.path.join(annotations_dir, xml_file))
    root = tree.getroot()

    size = root.find('size')
    width = int(size.find('width').text)
    height = int(size.find('height').text)

    yolo_lines = []
    for obj in root.findall('object'):
        cls_name = obj.find('name').text
        if cls_name not in class_to_id:
            continue
        cls_id = class_to_id[cls_name]

        bndbox = obj.find('bndbox')
        xmin = int(bndbox.find('xmin').text)
        ymin = int(bndbox.find('ymin').text)
        xmax = int(bndbox.find('xmax').text)
        ymax = int(bndbox.find('ymax').text)

        # 转换为YOLO格式 (归一化)
        x_center = (xmin + xmax) / 2.0 / width
        y_center = (ymin + ymax) / 2.0 / height
        bbox_width = (xmax - xmin) / width
        bbox_height = (ymax - ymin) / height

        yolo_lines.append(f'{cls_id} {x_center:.6f} {y_center:.6f} {bbox_width:.6f} {bbox_height:.6f}')

    # 保存为txt
    txt_name = xml_file.replace('.xml', '.txt')
    with open(os.path.join(labels_dir, txt_name), 'w') as f:
        f.write('\n'.join(yolo_lines))

print(f'转换完成，共处理 {len(xml_files)} 个标注文件')

类别映射: {'helmet': 0, 'head': 1, 'person': 2}
转换完成，共处理 5000 个标注文件


In [3]:
# 数据集划分
import os
import shutil
import random

random.seed(42)

images_dir = "helmet/images"
labels_dir = "helmet/labels"

# 获取所有图片名（不含扩展名）
image_files = [f for f in os.listdir(images_dir) if f.endswith(".png")]
image_stems = [os.path.splitext(f)[0] for f in image_files]
random.shuffle(image_stems)

# 找出含person类(类别ID=2)的图片，用于过采样
person_stems = []
for stem in image_stems:
    label_file = os.path.join(labels_dir, f"{stem}.txt")
    if os.path.exists(label_file):
        with open(label_file) as f:
            for line in f:
                if line.startswith("2 "):  # person类
                    person_stems.append(stem)
                    break
print(f"含person类图片: {len(person_stems)} 张")

# 划分数据集 8:2
split_ratio = 0.8
split_idx = int(len(image_stems) * split_ratio)
train_stems = image_stems[:split_idx]
val_stems = image_stems[split_idx:]

# 对训练集中含person的图片过采样3倍
person_in_train = [s for s in person_stems if s in train_stems]
oversample_count = len(person_in_train) * 3
print(f"训练集中含person: {len(person_in_train)} 张, 过采样 +{oversample_count} 张")
print(f"训练集: {len(train_stems)} + {oversample_count} = {len(train_stems) + oversample_count} 张")
print(f"验证集: {len(val_stems)} 张")

# 创建目录结构
for subset in ["train", "val"]:
    os.makedirs(f"dataset/{subset}/images", exist_ok=True)
    os.makedirs(f"dataset/{subset}/labels", exist_ok=True)

# 复制文件
def copy_files(stems, subset):
    for stem in stems:
        img_src = os.path.join(images_dir, f"{stem}.png")
        lbl_src = os.path.join(labels_dir, f"{stem}.txt")
        img_dst = os.path.join(f"dataset/{subset}/images", f"{stem}.png")
        lbl_dst = os.path.join(f"dataset/{subset}/labels", f"{stem}.txt")
        if os.path.exists(img_src):
            shutil.copy2(img_src, img_dst)
        if os.path.exists(lbl_src):
            shutil.copy2(lbl_src, lbl_dst)

copy_files(train_stems, "train")
copy_files(val_stems, "val")

# 对训练集含person图片过采样: 为每张生成3个副本
for stem in person_in_train:
    for dup_idx in range(1, 4):
        new_stem = f"{stem}_dup{dup_idx}"
        img_src = os.path.join(images_dir, f"{stem}.png")
        lbl_src = os.path.join(labels_dir, f"{stem}.txt")
        img_dst = os.path.join("dataset/train/images", f"{new_stem}.png")
        lbl_dst = os.path.join("dataset/train/labels", f"{new_stem}.txt")
        shutil.copy2(img_src, img_dst)
        shutil.copy2(lbl_src, lbl_dst)

print("数据集划分完成")


含person类图片: 303 张
训练集中含person: 246 张, 过采样 +738 张
训练集: 4000 + 738 = 4738 张
验证集: 1000 张
数据集划分完成


In [4]:
# data.yaml配置
import yaml

data_yaml = {
    'path': 'dataset',
    'train': 'train/images',
    'val': 'val/images',
    'nc': 3,
    'names': ['helmet', 'head', 'person']
}

with open('data.yaml', 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True)

print('data.yaml 配置完成:')
print(yaml.dump(data_yaml, default_flow_style=False, allow_unicode=True))

data.yaml 配置完成:
names:
- helmet
- head
- person
nc: 3
path: dataset
train: train/images
val: val/images



In [5]:
# cpu/GPU yolov8训练
# 根据本机相关配置动态调整训练参数
import torch
from ultralytics import YOLO

# 检测设备
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'使用设备: {device}')

# 根据设备动态调整训练参数
if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU: {gpu_name}')
    print(f'显存: {gpu_mem:.1f} GB')
    batch_size = 32
    workers = 4
else:
    batch_size = 4
    workers = 2

print(f'Batch Size: {batch_size}, Workers: {workers}')

# 加载预训练模型
model = YOLO('yolov8n.pt')

# 训练
results = model.train(
    data='data.yaml',
    epochs=50,
    imgsz=416,
    batch=batch_size,
    device=device,
    workers=workers,
    project='runs',
    name='helmet_detection',
    exist_ok=True
)

使用设备: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
显存: 6.0 GB
Batch Size: 32, Workers: 4
Ultralytics 8.4.53  Python-3.12.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=helmet_detecti

In [6]:
# 评估mAP@50%
import os
from ultralytics import YOLO

model_path = "runs/detect/runs/helmet_detection/weights/best.pt"
if not os.path.exists(model_path):
    print(f"模型文件不存在: {model_path}")
    print("请先运行 Cell 4 完成训练")
else:
    model = YOLO(model_path)
    metrics = model.val(data="data.yaml", imgsz=416)
    print(f"\nmAP@50:  {metrics.box.map50:.4f}")
    print(f"mAP@50-95: {metrics.box.map:.4f}")


Ultralytics 8.4.53  Python-3.12.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1301.3163.8 MB/s, size: 257.1 KB)
val: Scanning D:\program-codex\hw3\hw3\dataset\val\labels.cache... 1000 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1000/1000  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 12.6it/s 5.0s0.1s
                   all       1000       5138      0.638      0.579      0.617      0.403
                helmet        904       3734      0.934      0.881      0.941      0.611
                  head        189       1177      0.919      0.844      0.906      0.594
                person         57        227     0.0599     0.0132    0.00485    0.00301
Speed: 0.4ms preprocess, 1.8ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to D:\pr